# 근거 있는 안내문과 구조화 출력

보낼 수 있는 안내문을 만들려면 모델이 반환한 문장과 JSON을 각각 어디까지 믿을 수 있는지 판단해야 한다.

### 노트북 실행 준비

Python 3.12 커널에서 위쪽 셀부터 실행하세요. 패키지가 없는 커널에서는 새 코드 셀에 다음 명령을 실행한 뒤 예제로 이동하세요.

%pip install "pydantic>=2,<3"

## 형식 검사 뒤에 남는 질문

모델이 날짜와 신청 대상을 빠짐없이 적었다고 해도 원문에 없는 날짜를 채웠을 수 있다. 형식 검사는 날짜가 들어갈 자리를 찾을 수 있게 하고, 내용 대조는 그 자리에 써도 될 값을 판정한다.

원문에는 “일반 회원 대상. 접수 시작일은 추후 공지”라고 적혀 있다. 날짜를 비워 두는 것과 이번 달 1일로 추정하는 것은 수신자에게 다른 행동을 요구한다.

값을 모르는 상태와 필드가 누락된 상태도 구별한다. 전자는 담당자가 다음 공지를 기다릴 수 있지만 후자는 전달 계약부터 다시 확인해야 한다.

## 스키마를 통과한 오답

날짜가 선택 값이어도 필드 자체는 필수다. 아래 반환값은 형식 검사를 통과한다.

In [1]:
from datetime import date
from pydantic import BaseModel
class Notice(BaseModel):
    audience: str
    start: date | None

candidate = Notice.model_validate({"audience": "일반 회원", "start": "2027-03-01"})
source = {"audience": "일반 회원", "start": None}
print(candidate.model_dump() == source)

False


False는 날짜를 파싱하지 못했다는 뜻이 아니다. 파싱에 성공한 날짜가 원문의 미정 상태와 다르다.

### 사람이 다시 확인할 필드 찾기

후보를 버리기 전에 어느 필드가 원문과 다른지 담당자에게 알려 준다. 두 필드가 모두 달라도 빠뜨리지 않는다.

In [2]:
actual = candidate.model_dump()

In [3]:
# 이 셀에 풀이를 작성하세요.

<details class="notebook-answer"><summary>풀이와 실행 결과</summary><pre><code>print([key for key in source if source[key] != actual[key]])</code></pre><pre>[&#x27;start&#x27;]
</pre><p>start 한 칸만 다르다. 같은 입력으로 프롬프트를 바꿨을 때도 이 목록을 비교하면 수정 효과를 확인할 수 있다.</p></details>

## 완성된 JSON이 오기 전에는 무엇을 하는가?

화면에 도착한 응답 조각이 날짜 필드 중간에서 끊겼다. 뒤 조각을 이어 붙이기 전에는 스키마 검증 결과를 확정할 수 없다.

In [4]:
import json
part = '{"audience":"일반 회원","start":'
try:
    json.loads(part)
except json.JSONDecodeError as error:
    print("다음 조각 대기", error.pos)
raw = part + 'null}'
print(Notice.model_validate_json(raw).model_dump())

다음 조각 대기 28
{'audience': '일반 회원', 'start': None}


null이 도착하면 미정 값을 보존한 공지가 된다. 수업에서 실제 스트림을 받을 때는 단순히 중괄호가 닫힌 모양만 보지 말고 응답 완료 이벤트와 중단 상태도 확인한다.

### 없는 날짜 필드와 명시적인 미정 나누기

누락한 필드는 SDK 재요청 또는 수동 확인으로 보내고, null인 경우에는 원문과 대조할 수 있는 값으로 보존한다.

In [5]:
from pydantic import ValidationError
missing = {"audience":"일반 회원"}

In [6]:
# 이 셀에 풀이를 작성하세요.

<details class="notebook-answer"><summary>풀이와 실행 결과</summary><pre><code>try:
    Notice.model_validate(missing)
except ValidationError as error:
    print(error.errors()[0][&quot;loc&quot;], error.errors()[0][&quot;type&quot;])
print(Notice.model_validate_json(raw).start is None)</code></pre><pre>(&#x27;start&#x27;,) missing
True
</pre><p>필드 누락과 문서에서 미정이라고 한 상태를 별도로 기록하면 같은 재요청을 불필요하게 반복하지 않는다.</p></details>

## 다른 입력에 적용하기

같은 공지를 두 서비스에 넣고 요약을 한 문장씩 원문에 대응시킨다. 날짜가 실제로 확정된 새 공지를 교체해 넣고 이전 프롬프트와 스키마가 그 변화를 보존하는지 살핀다.

반환 문자열, 파싱한 값, 사람이 보류한 사유를 따로 남긴다. 스트리밍 조각만 받은 시점에 공지를 발송해도 되는지 마지막 이벤트와 함께 설명한다.

## 원문에서 공지 추출하기

시작일을 공지하지 않은 원문을 전달한다. 날짜를 만들어 냈는지 검사하려면 형식 통과 뒤에 원문을 대조해야 한다.

수업 커널에 langchain-openai와 langchain을 설치하고 OPENAI_API_KEY를 환경 변수에 설정한다. 호출에는 API 요금이 발생한다.

In [ ]:
import os
from langchain_openai import ChatOpenAI
model = ChatOpenAI(model=os.getenv("OPENAI_MODEL") or "gpt-4o-mini", timeout=45, max_retries=1)
notice = model.with_structured_output(Notice).invoke("일반 회원 대상 독서 모임입니다. 시작 날짜는 아직 미정입니다.")
print(notice.model_dump())
assert notice.start is None, "원문에 없는 날짜를 추출했습니다."

날짜가 명시된 공지로 바꾸고 날짜를 보존하는지 확인한다.